# 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import sqlite3
import warnings

warnings.filterwarnings("ignore")

# 2. Load Database

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
conn = sqlite3.connect("/content/drive/MyDrive/datasets/database.sqlite")

# 3. Extract EPL Match Data

In [ ]:
df = pd.read_sql("""
SELECT *
FROM Match
WHERE league_id = 1729
""", conn)

df.head()

,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
0,1729,1729,1729,2008/2009,1,2008-08-17 00:00:00,489042,10260,10261,1,...,10.00,1.28,5.5,12.00,1.30,4.75,10.0,1.29,4.50,11.00
1,1730,1729,1729,2008/2009,1,2008-08-16 00:00:00,489043,9825,8659,1,...,12.00,1.25,6.0,13.00,1.22,5.50,13.0,1.22,5.00,13.00
2,1731,1729,1729,2008/2009,1,2008-08-16 00:00:00,489044,8472,8650,0,...,1.73,5.50,3.8,1.65,5.00,3.40,1.7,4.50,3.40,1.73
3,1732,1729,1729,2008/2009,1,2008-08-16 00:00:00,489045,8654,8528,2,...,3.75,1.90,3.5,4.35,1.91,3.25,4.0,1.91,3.25,3.80
4,1733,1729,1729,2008/2009,1,2008-08-17 00:00:00,489046,10252,8456,4,...,3.75,1.90,3.5,4.35,1.91,3.25,4.0,1.91,3.30,3.75


# 4. Merge Team Names

####Load Team Table

In [ ]:
team = pd.read_sql("""
SELECT
team_api_id,
team_long_name
FROM Team
""", conn)

#### Merge Home Team

In [ ]:
df = df.merge(
    team,
    left_on='home_team_api_id',
    right_on='team_api_id',
    how='left'
)

df.rename(
    columns={'team_long_name':'home_team'},
    inplace=True
)

df.drop('team_api_id', axis=1, inplace=True)

#### Merge Away Team

In [ ]:
df = df.merge(
    team,
    left_on='away_team_api_id',
    right_on='team_api_id',
    how='left'
)

df.rename(
    columns={'team_long_name':'away_team'},
    inplace=True
)

df.drop('team_api_id', axis=1, inplace=True)

In [ ]:
df[['home_team','away_team','home_team_goal','away_team_goal']].head()

,home_team,away_team,home_team_goal,away_team_goal
0,Manchester United,Newcastle United,1,1
1,Arsenal,West Bromwich Albion,1,0
2,Sunderland,Liverpool,0,1
3,West Ham United,Wigan Athletic,2,1
4,Aston Villa,Manchester City,4,2


# 5. Feature Engineering Pipeline

#### Create Working DataFrame

In [ ]:
match_df = df.copy()

print(match_df.shape)

match_df.head()

(3040, 117)


,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA,home_team,away_team
0,1729,1729,1729,2008/2009,1,2008-08-17 00:00:00,489042,10260,10261,1,...,5.5,12.00,1.30,4.75,10.0,1.29,4.50,11.00,Manchester United,Newcastle United
1,1730,1729,1729,2008/2009,1,2008-08-16 00:00:00,489043,9825,8659,1,...,6.0,13.00,1.22,5.50,13.0,1.22,5.00,13.00,Arsenal,West Bromwich Albion
2,1731,1729,1729,2008/2009,1,2008-08-16 00:00:00,489044,8472,8650,0,...,3.8,1.65,5.00,3.40,1.7,4.50,3.40,1.73,Sunderland,Liverpool
3,1732,1729,1729,2008/2009,1,2008-08-16 00:00:00,489045,8654,8528,2,...,3.5,4.35,1.91,3.25,4.0,1.91,3.25,3.80,West Ham United,Wigan Athletic
4,1733,1729,1729,2008/2009,1,2008-08-17 00:00:00,489046,10252,8456,4,...,3.5,4.35,1.91,3.25,4.0,1.91,3.30,3.75,Aston Villa,Manchester City


In [ ]:
match_df = match_df[
    [
        'season',
        'stage',
        'date',

        'home_team',
        'away_team',

        'home_team_goal',
        'away_team_goal'
    ]
]

match_df.head()

,season,stage,date,home_team,away_team,home_team_goal,away_team_goal
0,2008/2009,1,2008-08-17 00:00:00,Manchester United,Newcastle United,1,1
1,2008/2009,1,2008-08-16 00:00:00,Arsenal,West Bromwich Albion,1,0
2,2008/2009,1,2008-08-16 00:00:00,Sunderland,Liverpool,0,1
3,2008/2009,1,2008-08-16 00:00:00,West Ham United,Wigan Athletic,2,1
4,2008/2009,1,2008-08-17 00:00:00,Aston Villa,Manchester City,4,2


In [ ]:
match_df['date'] = pd.to_datetime(match_df['date'])

match_df = match_df.sort_values('date').reset_index(drop=True)

In [ ]:
match_df.isnull().sum()

,0
season,0
stage,0
date,0
home_team,0
away_team,0
home_team_goal,0
away_team_goal,0


#### Save Clean Dataset

In [ ]:
match_df.to_csv(
    "clean_epl_matches.csv",
    index=False
)

print("Dataset Saved Successfully")

Dataset Saved Successfully


#### Create Historical Team Statistics

In [ ]:
def safe_mean(values):
    """
    Returns the average of a list.
    If the list is empty, returns 0.
    """
    if len(values) == 0:
        return 0
    return np.mean(values)

In [ ]:
match_df["home_avg_goals"] = 0.0
match_df["away_avg_goals"] = 0.0

match_df["home_avg_conceded"] = 0.0
match_df["away_avg_conceded"] = 0.0

In [ ]:
team_stats = {}

#### Create Team Statistics

In [ ]:
# Home team statistics
home_stats = match_df.groupby('home_team').agg(
    home_matches=('home_team', 'count'),
    home_goals_scored=('home_team_goal', 'sum'),
    home_goals_conceded=('away_team_goal', 'sum')
)

home_stats['avg_home_goals'] = (
    home_stats['home_goals_scored'] / home_stats['home_matches']
)

home_stats['avg_home_conceded'] = (
    home_stats['home_goals_conceded'] / home_stats['home_matches']
)

home_stats.head()

,home_matches,home_goals_scored,home_goals_conceded,avg_home_goals,avg_home_conceded
home_team,,,,,
Arsenal,152,306,122,2.013158,0.802632
Aston Villa,152,179,198,1.177632,1.302632
Birmingham City,38,38,35,1.000000,0.921053
Blackburn Rovers,76,98,90,1.289474,1.184211
Blackpool,19,30,37,1.578947,1.947368


#### Away team statistics

In [ ]:
away_stats = match_df.groupby('away_team').agg(
    away_matches=('away_team', 'count'),
    away_goals_scored=('away_team_goal', 'sum'),
    away_goals_conceded=('home_team_goal', 'sum')
)

away_stats['avg_away_goals'] = (
    away_stats['away_goals_scored'] / away_stats['away_matches']
)

away_stats['avg_away_conceded'] = (
    away_stats['away_goals_conceded'] / away_stats['away_matches']
)

away_stats.head()

,away_matches,away_goals_scored,away_goals_conceded,avg_away_goals,avg_away_conceded
away_team,,,,,
Arsenal,152,267,198,1.756579,1.302632
Aston Villa,152,156,264,1.026316,1.736842
Birmingham City,38,37,70,0.973684,1.842105
Blackburn Rovers,76,77,162,1.013158,2.131579
Blackpool,19,25,41,1.315789,2.157895


#### Merge home statistics

In [ ]:
match_df = match_df.merge(
    home_stats[
        ['avg_home_goals', 'avg_home_conceded']
    ],
    left_on='home_team',
    right_index=True,
    how='left'
)

#### Merge away statistics

In [ ]:
match_df = match_df.merge(
    away_stats[
        ['avg_away_goals', 'avg_away_conceded']
    ],
    left_on='away_team',
    right_index=True,
    how='left'
)

In [ ]:
match_df.head()

,season,stage,date,home_team,away_team,home_team_goal,away_team_goal,home_avg_goals,away_avg_goals,home_avg_conceded,away_avg_conceded,avg_home_goals,avg_home_conceded,avg_away_goals,avg_away_conceded
0,2008/2009,1,2008-08-16,Arsenal,West Bromwich Albion,1,0,0.0,0.0,0.0,0.0,2.013158,0.802632,0.962406,1.624060
1,2008/2009,1,2008-08-16,Sunderland,Liverpool,0,1,0.0,0.0,0.0,0.0,1.210526,1.190789,1.559211,1.335526
2,2008/2009,1,2008-08-16,West Ham United,Wigan Athletic,2,1,0.0,0.0,0.0,0.0,1.466165,1.308271,0.989474,1.873684
3,2008/2009,1,2008-08-16,Everton,Blackburn Rovers,2,3,0.0,0.0,0.0,0.0,1.697368,1.092105,1.013158,2.131579
4,2008/2009,1,2008-08-16,Middlesbrough,Tottenham Hotspur,2,1,0.0,0.0,0.0,0.0,0.894737,1.052632,1.486842,1.447368


In [ ]:
print(match_df.columns.tolist())

['season', 'stage', 'date', 'home_team', 'away_team', 'home_team_goal', 'away_team_goal', 'home_avg_goals', 'away_avg_goals', 'home_avg_conceded', 'away_avg_conceded', 'avg_home_goals', 'avg_home_conceded', 'avg_away_goals', 'avg_away_conceded']


In [ ]:
match_df = match_df[
    [
        'season',
        'stage',
        'date',
        'home_team',
        'away_team',
        'home_team_goal',
        'away_team_goal'
    ]
].copy()

In [ ]:
print(match_df.columns.tolist())

['season', 'stage', 'date', 'home_team', 'away_team', 'home_team_goal', 'away_team_goal']


#### Rolling Team Statistics

In [ ]:
match_df['date'] = pd.to_datetime(match_df['date'])

match_df = match_df.sort_values('date').reset_index(drop=True)

In [ ]:
# Rolling average home goals (previous home matches only)
match_df["rolling_home_avg_goals"] = (
    match_df.groupby("home_team")["home_team_goal"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [ ]:
# First home match has no history
match_df["rolling_home_avg_goals"] = (
    match_df["rolling_home_avg_goals"].fillna(0)
)

In [ ]:
match_df[
    [
        "date",
        "home_team",
        "home_team_goal",
        "rolling_home_avg_goals"
    ]
].head(15)

,date,home_team,home_team_goal,rolling_home_avg_goals
0,2008-08-16,Arsenal,1,0.0
1,2008-08-16,Sunderland,0,0.0
2,2008-08-16,West Ham United,2,0.0
3,2008-08-16,Everton,2,0.0
4,2008-08-16,Middlesbrough,2,0.0
5,2008-08-16,Bolton Wanderers,3,0.0
6,2008-08-16,Hull City,2,0.0
7,2008-08-17,Manchester United,1,0.0
8,2008-08-17,Aston Villa,4,0.0
9,2008-08-17,Chelsea,4,0.0


In [ ]:
match_df[match_df["home_team"] == "Arsenal"][
    [
        "date",
        "home_team_goal",
        "rolling_home_avg_goals"
    ]
].head(10)

,date,home_team_goal,rolling_home_avg_goals
0,2008-08-16,1,0.000000
25,2008-08-30,3,1.000000
54,2008-09-27,1,2.000000
74,2008-10-18,3,1.666667
96,2008-10-29,4,2.000000
110,2008-11-08,2,2.400000
120,2008-11-15,0,2.333333
150,2008-12-06,1,2.000000
176,2008-12-21,1,1.875000
191,2008-12-28,1,1.777778


#### Rolling Away Goals

In [ ]:
match_df["rolling_away_avg_goals"] = (
    match_df.groupby("away_team")["away_team_goal"]
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0)
)

#### Rolling Home Conceded

In [ ]:
match_df["rolling_home_avg_conceded"] = (
    match_df.groupby("home_team")["away_team_goal"]
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0)
)

#### Rolling Away Conceded

In [ ]:
match_df["rolling_away_avg_conceded"] = (
    match_df.groupby("away_team")["home_team_goal"]
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0)
)

In [ ]:
match_df[
[
'home_team',
'away_team',
'rolling_home_avg_goals',
'rolling_away_avg_goals',
'rolling_home_avg_conceded',
'rolling_away_avg_conceded'
]
].head(15)

,home_team,away_team,rolling_home_avg_goals,rolling_away_avg_goals,rolling_home_avg_conceded,rolling_away_avg_conceded
0,Arsenal,West Bromwich Albion,0.0,0.0,0.0,0.0
1,Sunderland,Liverpool,0.0,0.0,0.0,0.0
2,West Ham United,Wigan Athletic,0.0,0.0,0.0,0.0
3,Everton,Blackburn Rovers,0.0,0.0,0.0,0.0
4,Middlesbrough,Tottenham Hotspur,0.0,0.0,0.0,0.0
5,Bolton Wanderers,Stoke City,0.0,0.0,0.0,0.0
6,Hull City,Fulham,0.0,0.0,0.0,0.0
7,Manchester United,Newcastle United,0.0,0.0,0.0,0.0
8,Aston Villa,Manchester City,0.0,0.0,0.0,0.0
9,Chelsea,Portsmouth,0.0,0.0,0.0,0.0


In [ ]:
match_df["rolling_away_avg_goals"] = (
    match_df.groupby("away_team")["away_team_goal"]
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0)
)

In [ ]:
match_df["rolling_home_avg_conceded"] = (
    match_df.groupby("home_team")["away_team_goal"]
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0)
)

In [ ]:
match_df["rolling_away_avg_conceded"] = (
    match_df.groupby("away_team")["home_team_goal"]
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0)
)

In [ ]:
match_df.head()

,season,stage,date,home_team,away_team,home_team_goal,away_team_goal,rolling_home_avg_goals,rolling_away_avg_goals,rolling_home_avg_conceded,rolling_away_avg_conceded
0,2008/2009,1,2008-08-16,Arsenal,West Bromwich Albion,1,0,0.0,0.0,0.0,0.0
1,2008/2009,1,2008-08-16,Sunderland,Liverpool,0,1,0.0,0.0,0.0,0.0
2,2008/2009,1,2008-08-16,West Ham United,Wigan Athletic,2,1,0.0,0.0,0.0,0.0
3,2008/2009,1,2008-08-16,Everton,Blackburn Rovers,2,3,0.0,0.0,0.0,0.0
4,2008/2009,1,2008-08-16,Middlesbrough,Tottenham Hotspur,2,1,0.0,0.0,0.0,0.0


In [ ]:
match_df.to_csv("match_features_v1.csv", index=False)

print("Feature dataset created successfully.")

Feature dataset created successfully.


In [ ]:
team_stats = match_df.groupby("home_team").agg({
    "rolling_home_avg_goals": "last",
    "rolling_home_avg_conceded": "last"
})

away_stats = match_df.groupby("away_team").agg({
    "rolling_away_avg_goals": "last",
    "rolling_away_avg_conceded": "last"
})

team_stats = team_stats.join(away_stats, how="outer")
team_stats = team_stats.fillna(0)

team_stats.to_csv("team_stats.csv")

print("team_stats.csv saved successfully")

team_stats.csv saved successfully


In [ ]:
from sklearn.preprocessing import LabelEncoder
home_encoder = LabelEncoder()
away_encoder = LabelEncoder()
season_encoder = LabelEncoder()

match_df["home_team"] = home_encoder.fit_transform(match_df["home_team"])
match_df["away_team"] = away_encoder.fit_transform(match_df["away_team"])
match_df["season"] = season_encoder.fit_transform(match_df["season"])

In [ ]:
X = match_df[
    [
        "season",
        "stage",
        "home_team",
        "away_team",
        "rolling_home_avg_goals",
        "rolling_away_avg_goals",
        "rolling_home_avg_conceded",
        "rolling_away_avg_conceded"
    ]
]

y = match_df[
    [
        "home_team_goal",
        "away_team_goal"
    ]
]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(2432, 8)
(608, 8)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42

)

model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=300, random_state=42)

In [ ]:
pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

print("MAE :", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(((y_test - pred) ** 2).mean().mean()))
print("R2 :", r2_score(y_test, pred))

MAE : 0.9646929824561405
RMSE: 1.2232496595266904
R2 : -0.00804453183870435


In [ ]:
import joblib

In [ ]:
joblib.dump(model, "match_prediction_model.pkl")

['match_prediction_model.pkl']

In [ ]:
joblib.dump(home_encoder, "home_encoder.pkl")
joblib.dump(away_encoder, "away_encoder.pkl")
joblib.dump(season_encoder, "season_encoder.pkl")

['season_encoder.pkl']

In [ ]:
feature_columns = X.columns.tolist()

joblib.dump(feature_columns, "feature_columns.pkl")

['feature_columns.pkl']

In [ ]:
print("All files saved successfully.")

All files saved successfully.
